# Fashion Mnist Cnn mit Weights & Biases

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/week_06/03_fminst_cnn_wb.ipynb)

This notebook contains an exercise from the Deep Learning, Machine Learning, and AI course.

## Einführung in Weights & Biases (W&B)

Weights & Biases ist ein Tool zum Tracking von Experimenten, Visualisierung von Metriken und Verwalten von Modellen. In diesem Notebook werden wir W&B nutzen, um den Trainingsprozess zu überwachen und die Hyperparameter sowie Daten zu protokollieren.

### Schritt 1: Konto erstellen
Gehen Sie auf [wandb.ai](https://wandb.ai/site) und erstellen Sie ein kostenloses Konto.

### Schritt 2: API Key abrufen
Nachdem Sie sich eingeloggt haben, finden Sie Ihren API-Key in den [Benutzereinstellungen](https://wandb.ai/authorize).

### Schritt 3: Einloggen im Notebook
Wenn Sie `wandb.init()` aufrufen, werden Sie aufgefordert, Ihren API-Key einzugeben, falls Sie noch nicht eingeloggt sind.

In [ ]:
!pip install wandb -qU

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from functools import partial
import numpy as np

# 1. Fashion MNIST Datensatz laden
fashion_mnist = keras.datasets.fashion_mnist
(x_train_full, y_train_full), (x_test_full, y_test_full) = fashion_mnist.load_data()

# 2. Vorverarbeitung
# Normalisierung und Hinzufügen der Kanal-Dimension
x_train_full = x_train_full.astype('float32') / 255.0
x_test = x_test_full.astype('float32') / 255.0
x_train_full = x_train_full[..., np.newaxis]
x_test = x_test[..., np.newaxis]
y_test = y_test_full

# Validierungsset erstellen (erste 5000 Datenpunkte)
x_valid, x_train = x_train_full[:5000], x_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

In [ ]:
import wandb
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint, WandbEvalCallback
import os

# W&B initialisieren
run = wandb.init(project="fashion-mnist-cnn",
                 config={
                     "learning_rate": 0.001,
                     "epochs": 10,
                     "batch_size": 32,
                     "architecture": "CNN",
                     "dataset": "Fashion-MNIST",
                     "optimizer": "nadam",
                     "conv_1_filters": 64,
                     "conv_1_kernel": 7,
                     "conv_2_filters": 128,
                     "conv_3_filters": 128,
                     "conv_4_filters": 256,
                     "conv_5_filters": 256,
                     "dense_1_units": 128,
                     "dense_2_units": 64,
                     "dropout_rate": 0.5
                 })
config = wandb.config

# Daten als Artifacts loggen
def log_data_as_artifact(x, y, name, type):
    filename = f"{name}.npz"
    np.savez(filename, x=x, y=y)
    artifact = wandb.Artifact(name, type=type)
    artifact.add_file(filename)
    wandb.log_artifact(artifact)

log_data_as_artifact(x_train, y_train, "train_data", "dataset")
log_data_as_artifact(x_valid, y_valid, "valid_data", "dataset")


In [ ]:
# 3. Modell erstellen (Modernes CNN mit He-Initialisierung)
tf.random.set_seed(42)

DefaultConv2D = partial(tf.keras.layers.Conv2D, kernel_size=3, padding="same",
                        activation="relu", kernel_initializer="he_normal")

model = tf.keras.Sequential([
    DefaultConv2D(filters=config.conv_1_filters, kernel_size=config.conv_1_kernel, input_shape=[28, 28, 1]),
    tf.keras.layers.MaxPool2D(),
    DefaultConv2D(filters=config.conv_2_filters),
    DefaultConv2D(filters=config.conv_3_filters),
    tf.keras.layers.MaxPool2D(),
    DefaultConv2D(filters=config.conv_4_filters),
    DefaultConv2D(filters=config.conv_5_filters),
    tf.keras.layers.MaxPool2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=config.dense_1_units, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dropout(config.dropout_rate),
    tf.keras.layers.Dense(units=config.dense_2_units, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dropout(config.dropout_rate),
    tf.keras.layers.Dense(units=10, activation="softmax")
])

# Layer Details in W&B Config loggen
layer_details = []
for layer in model.layers:
    details = {"name": layer.name, "class": layer.__class__.__name__}
    if hasattr(layer, 'filters'): details['filters'] = layer.filters
    if hasattr(layer, 'kernel_size'): details['kernel_size'] = layer.kernel_size
    if hasattr(layer, 'units'): details['units'] = layer.units
    layer_details.append(details)

wandb.config.update({"layers": layer_details}, allow_val_change=True)

# 4. Modellzusammenfassung ausgeben
print(model.summary())

# 5. Modell kompilieren
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=tf.keras.optimizers.Nadam(learning_rate=config.learning_rate),
              metrics=["accuracy"])


In [ ]:
from wandb.integration.keras import WandbMetricsLogger

# 6. Modell trainieren
history = model.fit(x_train, y_train, epochs=config.epochs,
                    validation_data=(x_valid, y_valid),
                    callbacks=[WandbMetricsLogger(), WandbModelCheckpoint(filepath='model.keras')])


In [ ]:
# Trainings- und Validierungsverlust plotten
plt.plot(history.history['loss'], label='Trainingsverlust')
plt.plot(history.history['val_loss'], label='Validierungsverlust')
plt.title('Trainings- und Validierungsverlust')
plt.xlabel('Epochen')
plt.ylabel('Verlust')
plt.legend()
plt.show()

In [ ]:
# 7. Evaluierung auf dem Testdatensatz
print("\nEvaluierung auf dem Testdatensatz:")
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test-Genauigkeit: {test_acc}")

# 8. Vorhersagen für die ersten 3 Datenpunkte des Testsets
x_new = x_test[:3]

# predict aufrufen
y_proba = model.predict(x_new)
print("\nVorhersagewahrscheinlichkeiten (predict):")
print(y_proba.round(2))

# Vorhergesagte Klassen bestimmen
y_pred = np.argmax(model.predict(x_new), axis=-1)
print("\nVorhergesagte Klassen:")
print(y_pred)
print("Tatsächliche Klassen:")
print(y_test[:3])

In [ ]:
wandb.finish()